In [ ]:
"""Mobile Sales Dataset - Section 2 solutions (Pappu Saha, BWU/MCA/25/003)"""
import pandas as pd, numpy as np, json
pd.set_option('display.width',200); pd.set_option('display.max_columns',20)
df = pd.read_csv('/content/drive/MyDrive/mobile dataset/mobile.csv')
df['Brands']=df['Brands'].str.strip();
df['Models']=df['Models'].str.strip();
df['Mobile']=df['Mobile'].str.strip()
df['Memory']=pd.to_numeric(df['Memory'],errors='coerce')
df['Discount %']=df['Discount']/df['Original Price']*100
# Data cleaning note: Storage 1 = 1 TB; values 2/4/8 (<16) are MB-era feature phones; odd values (10,48,89,100,129,130,140,153) dropped from storage analyses
df.loc[df.Storage==1,'Storage']=1024
STD=[16,32,64,128,256,512,1024]
clean=df[df.Storage.isin(STD)&df.Memory.between(1,32)&(df.Memory<=df.Storage)].copy()
R={}
# Q1
q1=df.groupby('Brands')['Models'].nunique().sort_values(ascending=False)
print('Q1\n',q1); R['q1']=q1.to_dict()
# Q2
q2=df.groupby('Mobile')['Selling Price'].max().sort_values(ascending=False).head(10)
print('Q2\n',q2); R['q2']=q2.to_dict()
# Q3
q3=df.groupby('Brands')['Selling Price'].mean().round(0).sort_values(ascending=False)
print('Q3\n',q3); R['q3']=q3.to_dict()
# Q4
q4=df.groupby('Brands').agg(avg_disc=('Discount','mean'),avg_pct=('Discount %','mean')).round(2).sort_values('avg_disc',ascending=False)
print('Q4\n',q4); R['q4']=q4.reset_index().values.tolist()
# Q5
q5=df.groupby('Brands').agg(avg_rating=('Rating','mean'),avg_price=('Selling Price','mean'),n=('Rating','size')).round(2)
q5=q5[q5.n>=5].sort_values('avg_rating',ascending=False)
print('Q5\n',q5); R['q5']=q5.reset_index().values.tolist(); R['overall_price']=df['Selling Price'].mean()
# Q6
q6=clean.groupby('Storage').agg(avg_price=('Selling Price','mean'),n=('Storage','size')).round(0)
q6=q6[q6.n>=10]
print('Q6\n',q6); R['q6']=q6.reset_index().values.tolist()
R['q6corr']=clean['Storage'].corr(clean['Selling Price'])
print(R['q6corr'])
# Q7
d7=df.sort_values('Discount',ascending=False).drop_duplicates('Mobile').head(10)[['Mobile','Original Price','Selling Price','Discount','Discount %']]
print('Q7\n',d7); R['q7']=d7.round(1).values.tolist()
# Q8 - valid rows: memory present and <= storage
v=clean[(clean.Memory>=2)&(clean['Selling Price']>=5000)]  # smartphones only (drop feature phones with MB-unit data)
g=v.groupby(['Memory','Storage']).agg(price=('Selling Price','mean'),rating=('Rating','mean'),n=('Rating','size')).reset_index()
g=g[g.n>=10].copy()
mm=lambda s:(s-s.min())/(s.max()-s.min())
g['score']=0.5*mm(g.rating)+0.5*(1-mm(g.price))
g=g.sort_values('score',ascending=False).round(2)
print('Q8\n',g); R['q8']=g.head(8).values.tolist(); R['q8n']=len(g)
# Q9 value score
w=v.drop_duplicates('Mobile').copy()
w['vs']=(0.30*mm(1/w['Selling Price'].clip(lower=1)*0+ (1-mm(np.log(w['Selling Price']))))
         +0.20*mm(w['Discount %'])+0.15*mm(np.log2(w['Storage']))+0.15*mm(np.log2(w['Memory']))+0.20*mm(w['Rating']))
q9=w.sort_values('vs',ascending=False).head(10)[['Mobile','Selling Price','Discount %','Storage','Memory','Rating','vs']]
q9['vs']=(q9.vs*100).round(1); q9['Discount %']=q9['Discount %'].round(1)
print('Q9\n',q9); R['q9']=q9.values.tolist()
# Q10
df['Good']=(df.Rating>=4.3).astype(int); clean['Good']=(clean.Rating>=4.3).astype(int)
print('good share',df.Good.mean())
fac=['Selling Price','Discount','Discount %','Memory','Storage']
t=clean
corr=t[fac+['Rating']].corrwith(t['Good']).round(3)
corr.loc['Rating']=df['Rating'].corr(df['Good'])
print('Q10\n',corr)
gm=t.groupby('Good')[fac].mean().round(1); print(gm)
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(300,random_state=1,min_samples_leaf=5).fit(t[fac],t.Good)
imp=pd.Series(rf.feature_importances_,fac).sort_values(ascending=False).round(3); print(imp)
R['q10corr']=corr.round(3).to_dict(); R['q10gm']=gm.reset_index().values.tolist(); R['q10imp']=imp.to_dict(); R['goodshare']=df.Good.mean()
R['clean_rows']=len(clean); R['rows']=len(df); R['brands']=df.Brands.nunique(); R['dups']=int(df.duplicated().sum())
json.dump(R,open('results.json','w'),default=float,indent=1)

Q1
 Brands
SAMSUNG         210
Nokia            83
OPPO             77
LG               64
realme           62
GIONEE           60
vivo             59
Lenovo           55
Motorola         50
Xiaomi           49
Infinix          40
ASUS             33
HTC              29
Apple            25
POCO             14
Google Pixel     11
IQOO              1
Name: Models, dtype: int64
Q2
 Mobile
Apple iPhone 13 Pro Max      179900
SAMSUNG Fold 2 5G            169999
Apple iPhone 13 Pro          169900
SAMSUNG Galaxy Z Fold3 5G    157999
Apple iPhone 11 Pro Max      150800
SAMSUNG Galaxy Fold 2        149999
Apple iPhone 12 Pro Max      149900
Apple iPhone XS Max          144900
Apple iPhone 11 Pro          140300
Apple iPhone 12 Pro          139900
Name: Selling Price, dtype: int64
Q3
 Brands
Apple           81986.0
Google Pixel    61392.0
IQOO            37190.0
HTC             28314.0
SAMSUNG         24296.0
vivo            23866.0
ASUS            20327.0
LG              19323.0
OPPO          